# Sliding Window Inference (ST-GCN)

Take a full video's pose data and detect punches using sliding window classification with the trained ST-GCN model.

**Input:** Full-video `_pose_norm.npy` (no pre-segmentation)
**Output:** Timeline of predicted punches with start/end frames and classes
**Validation:** Compare predictions against ground truth annotations (if available)

In [1]:
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## Configuration

Paths, model selection, and sliding window parameters.

In [2]:
PROJECT_ROOT = Path("../../")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "no_hardware"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "no_hardware"
MODELS_DIR = PROJECT_ROOT / "models" / "stgcn"
ANNOTATIONS_CSV = METADATA_DIR / "annotations.csv"

# Model parameters (must match training)
T = 48
N_JOINTS = 9
N_CHANNELS = 3
N_FEATURES = N_JOINTS * N_CHANNELS
N_CLASSES = 5
CLASS_TO_IDX = {"cross": 0, "hook": 1, "jab": 2, "uppercut": 3, "no_punch": 4}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

# Sliding window parameters
WINDOW_STEP = 2              # frames between successive windows
CONFIDENCE_THRESHOLD = 0.7   # minimum softmax score to accept a prediction
MIN_PUNCH_LENGTH = 4         # minimum consecutive frames for a detected punch event (post-merge)
MAX_GAP_TO_MERGE = 4         # max frames between same-class predictions to merge into one event

# Pick the best ST-GCN model from training
# List available models and choose the most recent or best-named one
available_models = sorted(MODELS_DIR.glob("*_best.pt"))
print(f"Available ST-GCN models in {MODELS_DIR}:")
for m in available_models:
    print(f"  {m.name}")

# Choose the most recent model by default
MODEL_PATH = available_models[-1] if available_models else None
print(f"\nUsing model: {MODEL_PATH}")

Available ST-GCN models in ../../models/stgcn:
  stgcn_20260526_001159_best.pt
  stgcn_20260526_002229_fold1_subject01_best.pt
  stgcn_20260526_002229_fold2_subject02_best.pt
  stgcn_20260526_002229_fold3_subject03_best.pt
  stgcn_20260526_002229_fold4_subject04_best.pt
  stgcn_20260526_141617_best.pt
  stgcn_20260526_141816_fold1_subject01_best.pt
  stgcn_20260526_141816_fold2_subject02_best.pt
  stgcn_20260526_141816_fold3_subject03_best.pt
  stgcn_20260526_141816_fold4_subject04_best.pt
  stgcn_20260526_170312_best.pt
  stgcn_20260526_170312_fold1_subject01_best.pt
  stgcn_20260526_170312_fold2_subject02_best.pt
  stgcn_20260526_170312_fold3_subject03_best.pt
  stgcn_20260526_170312_fold4_subject04_best.pt

Using model: ../../models/stgcn/stgcn_20260526_170312_fold4_subject04_best.pt


## Dataset Format

Ensure the ST-GCN input format is (C, T, V) = (3, T, 9).

In [3]:
class PunchDataset(Dataset):
    """PyTorch Dataset for segmented punch clips.
    
    Loads clips from data/clips/no_hardware/{class}/*.npy and pads to T frames.
    """
    
    def __init__(self, annotations_df: pd.DataFrame, clips_dir: Path, T: int, augment: bool = True):
        self.df = annotations_df.reset_index(drop=True)
        self.clips_dir = clips_dir
        self.T = T
        self.augment = augment
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clip_path = self.clips_dir / row["class"] / f"{row['clip_id']}.npy"
        clip = np.load(clip_path).astype(np.float32)  # (frames, 9, 3)
        
        clip = self._pad_or_truncate(clip)  # (T, 9, 3)
        
        # Reshape to ST-GCN expected format: (C, T, V) = (3, T, 9)
        clip = clip.transpose(2, 0, 1)
        
        tensor = torch.from_numpy(clip)
        label = CLASS_TO_IDX[row["class"]]
        
        return tensor, label
    
    def _pad_or_truncate(self, clip: np.ndarray) -> np.ndarray:
        """Pad with last frame to T, or truncate if longer."""
        clip_len = clip.shape[0]
        if clip_len >= self.T:
            return clip[:self.T]
        
        if self.augment:
            offset = random.randint(0, self.T - clip_len)
            pre_pad = np.repeat(clip[:1], offset, axis=0)
            post_pad = np.repeat(clip[-1:], self.T - offset - clip_len, axis=0)
            return np.concatenate([pre_pad, clip, post_pad], axis=0)
        
        pad_amount = self.T - clip_len
        last_frame = clip[-1:].repeat(pad_amount, axis=0)
        return np.concatenate([clip, last_frame], axis=0)

## Graph Construction

Define the skeleton adjacency matrix for the 9 joints.

In [4]:
# Anatomical skeleton edges (joint index pairs)
SKELETON_EDGES = [
    (0, 1), (0, 2),       # nose -> shoulders
    (1, 3), (3, 5),       # left arm: shoulder -> elbow -> wrist
    (2, 4), (4, 6),       # right arm: shoulder -> elbow -> wrist
    (1, 7), (2, 8),       # shoulder -> hip
    (7, 8),               # hip-to-hip
]

def build_adjacency(edges: list, n_joints: int) -> np.ndarray:
    """Build normalized adjacency matrix A_hat = D^-0.5 (A + I) D^-0.5."""
    A = np.eye(n_joints)
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1
    
    D = np.sum(A, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    A_hat = D_inv_sqrt @ A @ D_inv_sqrt
    return A_hat

A_hat = build_adjacency(SKELETON_EDGES, N_JOINTS)
A_hat_tensor = torch.from_numpy(A_hat).float()

## Load ST-GCN Model

Re-create the ST-GCN architecture and load the trained weights.

In [5]:
class GraphConv(nn.Module):
    """Spatial graph convolution: X' = A_hat @ X @ W"""
    def __init__(self, in_channels: int, out_channels: int, A_hat: torch.Tensor):
        super().__init__()
        self.register_buffer("A_hat", A_hat)
        self.linear = nn.Linear(in_channels, out_channels)
    
    def forward(self, x):
        # x: (B, C, T, V)
        B, C, T, V = x.shape
        x = x.permute(0, 2, 3, 1)  # (B, T, V, C)
        x = torch.einsum("vu,btuc->btvc", self.A_hat, x)
        x = self.linear(x)
        x = x.permute(0, 3, 1, 2)  # (B, C', T, V)
        return x


class STGCNBlock(nn.Module):
    """Single ST-GCN block: spatial graph conv + temporal conv + residual."""
    def __init__(self, in_channels: int, out_channels: int, A_hat: torch.Tensor,
                 temporal_kernel: int = 9, stride: int = 1):
        super().__init__()
        self.gcn = GraphConv(in_channels, out_channels, A_hat)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels,
                      kernel_size=(temporal_kernel, 1),
                      stride=(stride, 1),
                      padding=(temporal_kernel // 2, 0)),
            nn.BatchNorm2d(out_channels),
        )
        if in_channels == out_channels and stride == 1:
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        res = self.residual(x)
        x = self.gcn(x)
        x = self.tcn(x)
        x = self.relu(x + res)
        return x


class STGCN(nn.Module):
    """Full ST-GCN model for punch classification."""
    def __init__(self, A_hat: torch.Tensor, n_classes: int = 4,
                 in_channels: int = 3, n_joints: int = 9):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(in_channels * n_joints)
        
        self.blocks = nn.ModuleList([
            STGCNBlock(in_channels, 64, A_hat),
            STGCNBlock(64, 64, A_hat),
            STGCNBlock(64, 128, A_hat, stride=2),
            STGCNBlock(128, 128, A_hat),
            STGCNBlock(128, 256, A_hat, stride=2),
            STGCNBlock(256, 256, A_hat),
        ])
        
        self.fc = nn.Linear(256, n_classes)
    
    def forward(self, x):
        # x: (B, C, T, V)
        B, C, T, V = x.shape
        x = x.permute(0, 1, 3, 2).contiguous().view(B, C * V, T)
        x = self.data_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2)
        
        for block in self.blocks:
            x = block(x)
        
        x = x.mean(dim=[2, 3])  # (B, C')
        x = self.fc(x)
        return x

# Load model
model = STGCN(A_hat_tensor, n_classes=N_CLASSES, in_channels=N_CHANNELS, n_joints=N_JOINTS).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f"Loaded model: {MODEL_PATH.name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loaded model: stgcn_20260526_170312_fold4_subject04_best.pt
Parameters: 1,724,667


## Sliding Window Inference Function

Given a full-video pose array, slide a window of size T across it and classify each window.

In [6]:
def sliding_window_inference(pose_array: np.ndarray, model, window_size: int = T,
                              step: int = WINDOW_STEP) -> dict:
    """Run sliding window classification on a full-video pose array.
    
    Args:
        pose_array: shape (total_frames, 9, 3)
        model: trained ST-GCN model
        window_size: T (frames per window)
        step: frames between successive windows
    
    Returns:
        dict with:
            window_centers: (n_windows,) center frame of each window
            predictions: (n_windows,) predicted class index
            confidences: (n_windows,) softmax probability of the predicted class
            all_probs: (n_windows, n_classes) full probability distributions
    """
    total_frames = pose_array.shape[0]
    
    if total_frames < window_size:
        raise ValueError(f"Video too short: {total_frames} frames, need at least {window_size}")
    
    window_starts = list(range(0, total_frames - window_size + 1, step))
    n_windows = len(window_starts)
    
    # Pre-compute all windows as a batch tensor
    windows = np.zeros((n_windows, window_size, N_JOINTS, N_CHANNELS), dtype=np.float32)
    for i, start in enumerate(window_starts):
        windows[i] = pose_array[start:start + window_size]
    
    # Convert to ST-GCN format: (n_windows, 3, T, 9)
    windows_stgcn = windows.transpose(0, 3, 1, 2)
    tensor = torch.from_numpy(windows_stgcn).to(DEVICE)
    
    # Batch inference
    with torch.no_grad():
        logits = model(tensor)  # (n_windows, n_classes)
        probs = F.softmax(logits, dim=1).cpu().numpy()
    
    predictions = probs.argmax(axis=1)
    confidences = probs.max(axis=1)
    window_centers = np.array(window_starts) + window_size // 2
    
    return {
        "window_starts": np.array(window_starts),
        "window_centers": window_centers,
        "predictions": predictions,
        "confidences": confidences,
        "all_probs": probs,
    }

## Post-Processing: Merge Predictions into Events

Convert per-window predictions into a clean timeline of detected punch events.

**Logic:**
1. Filter low-confidence predictions (treat as no_punch)
2. Group consecutive same-class predictions
3. Merge events of the same class separated by small gaps
4. Discard events shorter than `MIN_PUNCH_LENGTH`

In [7]:
def merge_predictions_to_events(inference_results: dict,
                                 confidence_threshold: float = CONFIDENCE_THRESHOLD,
                                 min_punch_length: int = MIN_PUNCH_LENGTH,
                                 max_gap_to_merge: int = MAX_GAP_TO_MERGE) -> list[dict]:
    """Convert sliding window predictions into clean detected events.
    
    Returns list of dicts:
        {start_frame, end_frame, class, confidence}
    """
    predictions = inference_results["predictions"].copy()
    confidences = inference_results["confidences"]
    window_starts = inference_results["window_starts"]
    window_size = T
    
    no_punch_idx = CLASS_TO_IDX["no_punch"]
    
    # Step 1: low-confidence -> no_punch
    low_conf_mask = confidences < confidence_threshold
    predictions[low_conf_mask] = no_punch_idx
    
    # Step 2: group consecutive same-class windows
    events = []
    if len(predictions) == 0:
        return events
    
    current_class = predictions[0]
    current_start_window = 0
    current_confs = [confidences[0]]
    
    for i in range(1, len(predictions)):
        if predictions[i] == current_class:
            current_confs.append(confidences[i])
        else:
            # End of current event
            event_start = window_starts[current_start_window]
            event_end = window_starts[i - 1] + window_size - 1
            events.append({
                "start_frame": int(event_start),
                "end_frame": int(event_end),
                "class": IDX_TO_CLASS[current_class],
                "confidence": float(np.mean(current_confs)),
            })
            current_class = predictions[i]
            current_start_window = i
            current_confs = [confidences[i]]
    
    # Don't forget the last event
    event_start = window_starts[current_start_window]
    event_end = window_starts[-1] + window_size - 1
    events.append({
        "start_frame": int(event_start),
        "end_frame": int(event_end),
        "class": IDX_TO_CLASS[current_class],
        "confidence": float(np.mean(current_confs)),
    })
    
    # Step 3: merge same-class events separated by small gaps
    merged = []
    for event in events:
        if merged and merged[-1]["class"] == event["class"] and \
           event["start_frame"] - merged[-1]["end_frame"] <= max_gap_to_merge:
            merged[-1]["end_frame"] = event["end_frame"]
            merged[-1]["confidence"] = (merged[-1]["confidence"] + event["confidence"]) / 2
        else:
            merged.append(event)
    
    # Step 4: filter out events shorter than MIN_PUNCH_LENGTH (only for non-no_punch)
    final = [
        e for e in merged
        if e["class"] == "no_punch" or (e["end_frame"] - e["start_frame"] + 1) >= min_punch_length
    ]
    
    # Step 5: filter out no_punch events from final output (we only care about detected punches)
    detected_punches = [e for e in final if e["class"] != "no_punch"]
    
    return detected_punches

## Run Inference on One Video

Pick a video and run the full pipeline: load pose -> slide -> predict -> merge -> visualize.

In [8]:
# Discover available processed videos
available_videos = []
for subject_dir in sorted(PROCESSED_ROOT.iterdir()):
    if subject_dir.is_dir():
        for npy_path in subject_dir.glob("*_pose_norm.npy"):
            available_videos.append({
                "subject_id": subject_dir.name,
                "video_stem": npy_path.stem.replace("_pose_norm", ""),
                "pose_path": npy_path,
            })

print(f"Found {len(available_videos)} processed videos")
print("\nFirst 10:")
for v in available_videos[:10]:
    print(f"  {v['subject_id']} / {v['video_stem']}")

Found 68 processed videos

First 10:
  subject01 / hook_1m_right
  subject01 / uppercut_1m_right
  subject01 / uppercut_2m_right
  subject01 / jab_2m_right
  subject01 / jab_1m_right
  subject01 / cross_2m_left
  subject01 / jab_1m_left
  subject01 / uppercut_1m_left
  subject01 / cross_1m_left
  subject01 / uppercut_2m_left


In [ ]:
# Single test video override
# Set this to True and provide the path to test on a single video outside the subject folders
USE_SINGLE_TEST_VIDEO = True

if USE_SINGLE_TEST_VIDEO:
    TEST_VIDEO_NAME = "test_video4"  # change this to your video stem (without _pose_norm.npy)
    test_video_path = PROCESSED_ROOT / "test" / f"{TEST_VIDEO_NAME}_pose_norm.npy"
    
    if not test_video_path.exists():
        raise FileNotFoundError(f"Test video pose file not found: {test_video_path}")
    
    test_video = {
        "subject_id": "test",
        "video_stem": TEST_VIDEO_NAME,
        "pose_path": test_video_path,
    }
    print(f"Using single test video: {test_video_path}")
else:
    # Fall through to picking by index from the discovered list
    TEST_VIDEO_IDX = 0
    test_video = available_videos[TEST_VIDEO_IDX]
    print(f"Testing on: {test_video['subject_id']} / {test_video['video_stem']}")

SyntaxError: invalid syntax (2933316181.py, line 14)

In [ ]:
# Choose video by index or by stem
if USE_SINGLE_TEST_VIDEO:
    print(f"Testing on single video: {test_video['subject_id']} / {test_video['video_stem']}")
else:
    TEST_VIDEO_IDX = 0  # change this index to test different videos
    test_video = available_videos[TEST_VIDEO_IDX]
    print(f"Testing on: {test_video['subject_id']} / {test_video['video_stem']}")

pose_array = np.load(test_video["pose_path"])
print(f"Pose shape: {pose_array.shape}")

# Run inference
results = sliding_window_inference(pose_array, model)
print(f"Total windows: {len(results['predictions'])}")

# Distribution of window-level predictions
unique, counts = np.unique(results["predictions"], return_counts=True)
print(f"\nWindow-level prediction distribution:")
for u, c in zip(unique, counts):
    print(f"  {IDX_TO_CLASS[u]}: {c} ({100*c/len(results['predictions']):.1f}%)")

# Post-process to events
detected_punches = merge_predictions_to_events(results)
print(f"\nDetected punches: {len(detected_punches)}")
for i, p in enumerate(detected_punches[:20]):
    print(f"  {i+1}. {p['class']:10s}  frames {p['start_frame']:4d}-{p['end_frame']:4d}  "
          f"(conf={p['confidence']:.3f})")

## Visualize Timeline

Plot the per-window predictions over time, with detected events highlighted.

In [ ]:
def plot_inference_timeline(results, detected_punches, ground_truth_events=None,
                             total_frames=None, title=""):
    """Plot the sliding window predictions and detected events over time."""
    fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
    
    class_colors = {
        "cross": "#1f77b4",
        "hook": "#ff7f0e",
        "jab": "#2ca02c",
        "uppercut": "#d62728",
        "no_punch": "#cccccc",
    }
    
    # Top: per-window confidence colored by predicted class
    window_centers = results["window_centers"]
    predictions = results["predictions"]
    confidences = results["confidences"]
    
    for cls_idx, cls_name in IDX_TO_CLASS.items():
        mask = predictions == cls_idx
        axes[0].scatter(window_centers[mask], confidences[mask],
                       color=class_colors[cls_name], label=cls_name, s=15, alpha=0.7)
    
    axes[0].set_ylabel("Window confidence")
    axes[0].set_title(f"Per-window predictions: {title}")
    axes[0].legend(loc="upper right", ncol=5, fontsize=9)
    axes[0].axhline(CONFIDENCE_THRESHOLD, color="red", linestyle="--", alpha=0.5,
                    label=f"Threshold {CONFIDENCE_THRESHOLD}")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(alpha=0.3)
    
    # Bottom: detected events vs ground truth
    y_pred = 1
    for event in detected_punches:
        rect = mpatches.Rectangle(
            (event["start_frame"], y_pred - 0.4),
            event["end_frame"] - event["start_frame"], 0.8,
            facecolor=class_colors[event["class"]], edgecolor="black", linewidth=0.5,
        )
        axes[1].add_patch(rect)
    axes[1].text(-0.02, y_pred, "Predicted", transform=axes[1].get_yaxis_transform(),
                 ha="right", va="center", fontsize=10)
    
    if ground_truth_events:
        y_gt = 0
        for event in ground_truth_events:
            rect = mpatches.Rectangle(
                (event["start_frame"], y_gt - 0.4),
                event["end_frame"] - event["start_frame"], 0.8,
                facecolor=class_colors[event["class"]], edgecolor="black", linewidth=0.5,
            )
            axes[1].add_patch(rect)
        axes[1].text(-0.02, y_gt, "Ground Truth", transform=axes[1].get_yaxis_transform(),
                     ha="right", va="center", fontsize=10)
    
    if total_frames:
        axes[1].set_xlim(0, total_frames)
    axes[1].set_ylim(-0.6, 1.6)
    axes[1].set_yticks([])
    axes[1].set_xlabel("Frame number")
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Look up ground truth from annotations.csv if available
df_annotations = pd.read_csv(ANNOTATIONS_CSV)

def get_ground_truth(video_stem: str) -> list[dict]:
    """Find annotations for a given video stem (handles Label Studio hash prefix)."""
    matches = df_annotations[
        df_annotations["video_filename"].str.contains(video_stem, regex=False)
    ]
    return [
        {"start_frame": int(r["start_frame"]),
         "end_frame": int(r["end_frame"]),
         "class": r["class"]}
        for _, r in matches.iterrows()
        if r["class"] != "no_punch"
    ]

gt_events = get_ground_truth(test_video["video_stem"])
plot_inference_timeline(results, detected_punches, gt_events,
                       total_frames=pose_array.shape[0],
                       title=f"{test_video['subject_id']} / {test_video['video_stem']}")

## Quantitative Evaluation

Compare detected events to ground truth.
- True Positive: detected event overlaps a ground truth event of the same class
- False Positive: detected event has no matching ground truth
- False Negative: ground truth event has no matching detection

In [ ]:
def compute_overlap(e1: dict, e2: dict) -> float:
    """Compute IoU between two events."""
    intersection = max(0, min(e1["end_frame"], e2["end_frame"]) -
                         max(e1["start_frame"], e2["start_frame"]) + 1)
    union = (max(e1["end_frame"], e2["end_frame"]) -
             min(e1["start_frame"], e2["start_frame"]) + 1)
    return intersection / union if union > 0 else 0


def evaluate_events(detected: list[dict], ground_truth: list[dict],
                    iou_threshold: float = 0.3) -> dict:
    """Match detected events to ground truth and compute TP/FP/FN."""
    tp = 0
    fp = 0
    matched_gt = set()
    
    for det in detected:
        best_iou = 0
        best_gt_idx = None
        for i, gt in enumerate(ground_truth):
            if i in matched_gt:
                continue
            if det["class"] != gt["class"]:
                continue
            iou = compute_overlap(det, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i
        
        if best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1
    
    fn = len(ground_truth) - len(matched_gt)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}


metrics = evaluate_events(detected_punches, gt_events)
print(f"Ground truth events: {len(gt_events)}")
print(f"Detected events:     {len(detected_punches)}")
print(f"\nTrue Positives:  {metrics['tp']}")
print(f"False Positives: {metrics['fp']}")
print(f"False Negatives: {metrics['fn']}")
print(f"\nPrecision: {metrics['precision']:.3f}")
print(f"Recall:    {metrics['recall']:.3f}")
print(f"F1 score:  {metrics['f1']:.3f}")

## Evaluate Across All Test Videos

Run inference on all videos for a held-out test subject and report aggregate metrics.

In [ ]:
if not USE_SINGLE_TEST_VIDEO:
    TEST_SUBJECT = "subject04"

    print(f"Evaluating inference on all videos for {TEST_SUBJECT}...\n")

    test_videos = [v for v in available_videos if v["subject_id"] == TEST_SUBJECT]
    print(f"Found {len(test_videos)} videos for {TEST_SUBJECT}")

    all_metrics = []
    total_tp = 0
    total_fp = 0
    total_fn = 0

    for v in test_videos:
        pose_array = np.load(v["pose_path"])
        results = sliding_window_inference(pose_array, model)
        detected = merge_predictions_to_events(results)
        gt = get_ground_truth(v["video_stem"])
        
        metrics = evaluate_events(detected, gt)
        metrics["video"] = v["video_stem"]
        all_metrics.append(metrics)
        
        total_tp += metrics["tp"]
        total_fp += metrics["fp"]
        total_fn += metrics["fn"]

    # Aggregate metrics
    total_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    total_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    total_f1 = (2 * total_precision * total_recall / (total_precision + total_recall)
                if (total_precision + total_recall) > 0 else 0)

    print(f"\n{'Video':<40s} {'TP':>4s} {'FP':>4s} {'FN':>4s} {'P':>6s} {'R':>6s} {'F1':>6s}")
    print("=" * 75)
    for m in all_metrics:
        print(f"{m['video']:<40s} {m['tp']:>4d} {m['fp']:>4d} {m['fn']:>4d} "
              f"{m['precision']:>6.3f} {m['recall']:>6.3f} {m['f1']:>6.3f}")

    print("=" * 75)
    print(f"{'AGGREGATE':<40s} {total_tp:>4d} {total_fp:>4d} {total_fn:>4d} "
          f"{total_precision:>6.3f} {total_recall:>6.3f} {total_f1:>6.3f}")
else:
    TEST_SUBJECT = None
    test_videos = []
    all_metrics = []
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_precision = 0.0
    total_recall = 0.0
    total_f1 = 0.0

In [ ]:
print("=" * 60)
print("SLIDING WINDOW INFERENCE SUMMARY")
print("=" * 60)
print(f"Model:               {MODEL_PATH.name}")
if USE_SINGLE_TEST_VIDEO:
    print(f"Test video:          {test_video['subject_id']} / {test_video['video_stem']}")
    print("Videos tested:       1")
else:
    print(f"Test subject:        {TEST_SUBJECT}")
    print(f"Videos tested:       {len(test_videos)}")
print(f"Window size (T):     {T}")
print(f"Window step:         {WINDOW_STEP}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Min punch length:    {MIN_PUNCH_LENGTH}")
if USE_SINGLE_TEST_VIDEO:
    print(f"\nPrecision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1 score:  {metrics['f1']:.4f}")
else:
    print(f"\nAggregate Precision: {total_precision:.4f}")
    print(f"Aggregate Recall:    {total_recall:.4f}")
    print(f"Aggregate F1 score:  {total_f1:.4f}")
print("=" * 60)

## Annotated Video Output

Generate an MP4 with predictions overlaid on each frame for presentation purposes.

- Class label and confidence shown top-left per frame
- Colored border when a punch is being detected
- One MP4 per test video

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
from src.video import load_video_frames, video_fps, video_frame_size

ANNOTATED_OUTPUT_DIR = PROJECT_ROOT / "data" / "annotated_videos"
ANNOTATED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_DISPLAY_FRAMES = 30
NO_PUNCH_GAP_FRAMES = 15

def find_source_video(video_stem: str) -> Path:
    """Find the source MP4/MOV for a given video stem."""
    raw_root = PROJECT_ROOT / "data" / "raw" / "no_hardware"
    for ext in ("*.mp4", "*.mov", "*.MOV", "*.MP4", "*.avi"):
        for f in raw_root.rglob(ext):
            if f.stem == video_stem:
                return f
    return None


def make_annotated_video(source_video: Path, output_path: Path,
                         results: dict, detected_punches: list[dict]):
    """Render an annotated MP4 with predictions overlaid."""
    
    class_colors_bgr = {
        "cross":    (180, 119, 31),
        "hook":     (14, 127, 255),
        "jab":      (44, 160, 44),
        "uppercut": (40, 39, 214),
        "no_punch": (200, 200, 200),
    }
    
    cap = cv2.VideoCapture(str(source_video))
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {source_video}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    fps = video_fps(source_video)
    height, width = video_frame_size(source_video)
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
    
    # Build per-frame border and label lookup
    frame_to_border_class = ["no_punch"] * total_frames
    frame_to_label_class = [None] * total_frames
    frame_to_label_conf = [0.0] * total_frames
    
    for event in sorted(detected_punches, key=lambda e: e["start_frame"]):
        for f in range(event["start_frame"], min(event["end_frame"] + 1, total_frames)):
            frame_to_border_class[f] = event["class"]
        midpoint = (event["start_frame"] + event["end_frame"]) // 2
        label_start = min(midpoint, total_frames - 1)
        label_end = min(label_start + LABEL_DISPLAY_FRAMES - 1, total_frames - 1)
        for f in range(label_start, label_end + 1):
            frame_to_label_class[f] = event["class"]
            frame_to_label_conf[f] = event["confidence"]
    
    last_label_end = -1
    for f in range(total_frames):
        if frame_to_label_class[f] is not None:
            last_label_end = f
            continue
        if f - last_label_end >= NO_PUNCH_GAP_FRAMES:
            frame_to_label_class[f] = "no_punch"
            frame_to_label_conf[f] = 0.0
    
    for frame_idx, frame in load_video_frames(
        source_video,
        skip_frames=0,
        max_frames=None,
        front_camera=False,
    ):
        if frame_idx < total_frames:
            border_label = frame_to_border_class[frame_idx]
            label = frame_to_label_class[frame_idx]
            conf = frame_to_label_conf[frame_idx]
        else:
            border_label = "no_punch"
            label = None
            conf = 0.0
        
        border_color = class_colors_bgr.get(border_label, (200, 200, 200))
        if border_label != "no_punch":
            border = 10
            cv2.rectangle(frame, (0, 0), (width - 1, height - 1), border_color, border)
        
        if label is not None:
            text_color = class_colors_bgr.get(label, (200, 200, 200))
            overlay_text = f"{label.upper()} ({conf:.2f})"
            cv2.rectangle(frame, (0, 0), (350, 60), (0, 0, 0), -1)
            cv2.putText(frame, overlay_text, (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, text_color, 2, cv2.LINE_AA)
        
        # Frame counter
        cv2.putText(frame, f"f={frame_idx}", (10, height - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        
        writer.write(frame)
    
    writer.release()
    print(f"Annotated video saved: {output_path}")
    print(f"Total frames written: {total_frames}")


# Generate annotated video for the currently loaded test video
source_video = find_source_video(test_video["video_stem"])
if source_video is None:
    print(f"Could not find source video file for: {test_video['video_stem']}")
    print(f"Looked in: {PROJECT_ROOT / 'data' / 'raw' / 'no_hardware'}")
else:
    output_path = ANNOTATED_OUTPUT_DIR / f"{test_video['video_stem']}_annotated.mp4"
    make_annotated_video(source_video, output_path, results, detected_punches)